## Introduction to Cross-Validation

- ### What is Cross-Validation ?
    - Technique used to assess how well a machine learning model generalizes to an independent dataset

- #### Types of Cross-Validation
  - **K-Fold Cross-Validation**
    - Splits the dataset into K folds of approximately equal size
    - The model is trained on K-1 Folds and Valisated on the remaining fold
    - This process is repeated K times, and the average performance is completed
  
  - **Stratified K-Fold**
    - Ensures the each fold maintains the same class distribution as the original dataset

  - **Leave-One-Out Cross-Validation (LOOCV)**
    - Uses a single data point  for validating and the rest for training
    - Repeats this process for all data points
    - Computationally expensive but provides the most robust evaluation

---

## Hyperparameter Tuning
- What is Hyperparameter tunining
  - Parameters that are not learned by the model but are set before training these hyperparametes is crucial for optimizing model performance

- ### Techniques for Hyperparameter Tunining 
  - Grid Search
    - Exhaustively searches over a  predefined hyperparameter space
    - Example: Testing all combinations of values for max_depth and learninng rate
  
  - Random Search
    - Randomly samples combination of hyperparameter from the predefined space
    - More efficientthat Grid search when the parameter space is large

  - Importance of Hyperparameter Tunining
    - Prevents overfitting and underfitting by selecting the best configuration
    - Enhances model performance by optimizing critical settings
    - Without tuining the model might not reach its optimal performace, leading to:
      - Underfitting: Model falls to capture the underlying patterns
      - Overfitting: Model Memorizes the training data and performs poorly on unseen data

# Hands-on ( Featuring and model performance)

- Performs end-to-end feature enginnering, model evaluation and hyperparameter tunining on a dataset
- Tasks:
  - 1. Perform feature enginnering
  - 2. Train and Evaluate Model
  - 3. Apply Grid Search for Hyperparameter tunining

In [24]:
# Importining Necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import cross_val_predict, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

In [25]:
# Loading Titanic dataset
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

In [26]:
# Select relevant features 
df = df[['Pclass','Sex','Age','Fare','Embarked','Survived']]

In [27]:
# Handle Missing values
df.fillna({'Age':df['Age'].median()},inplace=True)
df.fillna({'Embarked':df['Embarked'].mode()[0]},inplace=True)

In [28]:
# Define Features and Target
X = df.drop(columns = ['Survived'])
y = df['Survived']

In [29]:
# Apply feature scaling and encoding
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(),['Age','Fare']),
        ('cat',OneHotEncoder(),['Pclass','Sex','Embarked'])
    ]
)

# to get the preprocessed feature we will pass it from the preprocessor
X_preprocessed = preprocessor.fit_transform(X)



- `rf_scores = cross_val_score(rf_model, X_preprocessed, y, cv=5, scoring='accuracy')`:
    * This is the core of the evaluation process. It uses `cross_val_score` from scikit-learn.
    * `rf_model`: This is the estimator (the model) to be evaluated.
    * `X_preprocessed`: This is your feature data. The name suggests it has already undergone some preprocessing steps like scaling or one-hot encoding.
    * `y`: This is the target variable or the labels for your data.
    * `cv=5`: This specifies the number of folds for the cross-validation. In this case, the data is split into 5 parts. The model will be trained 5 times. Each time, a different fold is used as the test set, and the remaining 4 folds are used for training.
    * `scoring='accuracy'`: This tells the function to use classification accuracy as the metric to evaluate the model's performance on each fold.




In [30]:
# Train and Evaluate the Losgistic regression
log_model = LogisticRegression()
log_scores = cross_val_score(log_model, X_preprocessed,y,cv=5,scoring = 'accuracy')
print(f"Logistic Regression Accuracy:{log_scores.mean():.2f}")

Logistic Regression Accuracy:0.79


In [31]:
# Train and Evaluate the Random Forest model 
rf_model = RandomForestClassifier(random_state=42)
rf_scores = cross_val_score(rf_model,X_preprocessed,y,cv=5,scoring='accuracy')
print(f"Random forest accuracy:{rf_scores.mean():.2f}")

Random forest accuracy:0.81



* **`param_grid`**: The name itself suggests its purpose: a grid of parameters.
* **`'n_estimators'`**: This is a key representing a hyperparameter of a tree-based model, such as a Random Forest or Gradient Boosting machine. It refers to the number of decision trees in the forest or boosting ensemble. The values `[50, 100, 200]` are the different numbers of trees that you want to test.
* **`'max_depth'`**: This key represents another hyperparameter that controls the maximum depth of each tree. `None` means the nodes are expanded until all leaves are pure or until all leaves contain less than `min_samples_split` samples. `10` and `20` are other maximum depths to be tested.
* **`'min_samples_split'`**: This is a hyperparameter that sets the minimum number of samples required to split an internal node. The values `[2, 5, 10]` are the different thresholds to be tested.


In [32]:
# Define Hyperparameter grid
param_grid = {
    'n_estimators':[50,100, 200],
    'max_depth':[None,10,20],
    'min_samples_split':[2,5,10]
}

grid search is finding all possible combination of hyperparameter tuning, and finding the modst accurate one

In [33]:
# Perform Grid Search
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid = param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)

In [34]:
grid_search.fit(X_preprocessed, y)

,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'max_depth': [None, 10, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [50, 100, ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [35]:
# Display best hyperparameter and scores
print(f"Best Hypereparameter:{grid_search.best_params_}")
print(f"Best Accuracy:{grid_search.best_score_:.2f}")

Best Hypereparameter:{'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}
Best Accuracy:0.83


In [36]:
print(grid_search.cv_results_)

{'mean_fit_time': array([0.27807117, 0.64182153, 1.13110271, 0.26803269, 0.58620133,
       0.87700291, 0.34809322, 0.47365923, 0.99256201, 0.25393548,
       0.66834569, 1.23919153, 0.33303905, 0.64838781, 1.07877192,
       0.22617407, 0.52658296, 1.05752473, 0.34633408, 0.5671176 ,
       1.00909829, 0.26655545, 0.45850391, 0.99252572, 0.22256703,
       0.51259012, 0.82409143]), 'std_fit_time': array([0.04284685, 0.11990282, 0.18987766, 0.05869309, 0.15347422,
       0.19621671, 0.04160178, 0.13511798, 0.22623168, 0.13727425,
       0.26942372, 0.44488208, 0.14895398, 0.29519634, 0.42301533,
       0.03723405, 0.06775623, 0.21771935, 0.05854846, 0.11137155,
       0.16324279, 0.07438495, 0.07904657, 0.15098339, 0.0465388 ,
       0.11195071, 0.0764014 ]), 'mean_score_time': array([0.0102541 , 0.03321018, 0.05503874, 0.01359677, 0.02315035,
       0.03167782, 0.01483688, 0.01658974, 0.02933102, 0.00901618,
       0.05338497, 0.04044557, 0.01202874, 0.01783843, 0.03634582,
       0.0

In [ ]:
# ptiniting all the conbinations in the form of  proper manner
df = pd.DataFrame(grid_search.cv_results_)
df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_depth,param_min_samples_split,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.278071,0.042847,0.010254,0.001596,None,2,50,"{'max_depth': None, 'min_samples_split': 2, 'n...",0.787709,0.803371,0.842697,0.792135,0.820225,0.809227,0.020152,22
1,0.641822,0.119903,0.033210,0.011420,None,2,100,"{'max_depth': None, 'min_samples_split': 2, 'n...",0.793296,0.803371,0.837079,0.792135,0.814607,0.808097,0.016604,25
2,1.131103,0.189878,0.055039,0.015557,None,2,200,"{'max_depth': None, 'min_samples_split': 2, 'n...",0.787709,0.803371,0.825843,0.792135,0.820225,0.805857,0.015031,26
3,0.268033,0.058693,0.013597,0.002688,None,5,50,"{'max_depth': None, 'min_samples_split': 5, 'n...",0.810056,0.808989,0.837079,0.825843,0.848315,0.826056,0.015260,8
4,0.586201,0.153474,0.023150,0.010993,None,5,100,"{'max_depth': None, 'min_samples_split': 5, 'n...",0.815642,0.808989,0.842697,0.825843,0.848315,0.828297,0.015146,4
5,0.877003,0.196217,0.031678,0.002964,None,5,200,"{'max_depth': None, 'min_samples_split': 5, 'n...",0.821229,0.808989,0.837079,0.820225,0.842697,0.826044,0.012223,10
6,0.348093,0.041602,0.014837,0.009806,None,10,50,"{'max_depth': None, 'min_samples_split': 10, '...",0.798883,0.797753,0.870787,0.803371,0.848315,0.823821,0.030084,15
7,0.473659,0.135118,0.016590,0.001115,None,10,100,"{'max_depth': None, 'min_samples_split': 10, '...",0.798883,0.803371,0.859551,0.814607,0.842697,0.823821,0.023486,17
8,0.992562,0.226232,0.029331,0.004396,None,10,200,"{'max_depth': None, 'min_samples_split': 10, '...",0.804469,0.803371,0.865169,0.814607,0.848315,0.827186,0.025022,6
9,0.253935,0.137274,0.009016,0.001051,10,2,50,"{'max_depth': 10, 'min_samples_split': 2, 'n_e...",0.793296,0.814607,0.848315,0.825843,0.837079,0.823828,0.018955,14
